# GeoMech-LogML — Notebook 01: Synthetic Agbada Data Exploration

This notebook walks through the **physics-based synthetic generator** for Agbada-like
(Niger Delta) well logs: facies, overpressure, compaction, log response and the
*ground-truth* geomechanics (E_static, ν, UCS) that the ML models must recover.

**Prerequisite**: run from the repository root with the package importable
(`pip install -e .` or set `PYTHONPATH=.`).

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import geomech_logml
from geomech_logml.data.synthetic import SyntheticConfig, generate_dataset

cfg = SyntheticConfig(n_wells=6, depth_min_m=900, depth_max_m=4200, step_m=1.0, seed=42)
df = generate_dataset(cfg)
df.head()

## 1. Property ranges — must respect published Agbada context

In [ ]:
stats = df[['GR','RHOB','NPHI','RT','VP','E_STAT','NU_STAT','UCS']].describe().T
stats[['min','25%','50%','75%','max']].round(3)

## 2. Facies & logs for one well

The generator uses a 4-state Markov facies succession (sand → shaly sand → sandy shale → shale)
and forward-models GR, RHOB, NPHI, RT and VP from the underlying lithology/porosity/fluid state.

In [ ]:
w = df[df.WELL == 'AGB-01']
fig, axes = plt.subplots(1, 6, figsize=(15, 9), sharey=True)
tracks = [('GR','API'), ('RHOB','g/cc'), ('NPHI','v/v'), ('RT','ohm.m'), ('VP','m/s'), ('FACIES','code')]
for ax, (c, u) in zip(axes, tracks):
    ax.plot(w[c], w.DEPT, lw=0.6); ax.invert_yaxis(); ax.set_title(f'{c} ({u})')
axes[3].set_xscale('log')
plt.tight_layout(); plt.show()

## 3. Overpressure: effective stress controls porosity (undercompaction)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 6))
d = df[df.WELL == 'AGB-01']
for a, (c, t) in zip(ax, [('PP_MPA','Pore pressure (MPa)'), ('SIG_EFF_MPA','Eff. stress (MPa)'), ('PHI_TRUE','Porosity')]):
    a.scatter(d[c], d.DEPT, c=d.FACIES, s=2, cmap='viridis'); a.invert_yaxis(); a.set_title(t)
hydro = 0.0101 * d.DEPT
ax[0].plot(hydro, d.DEPT, 'r--', label='hydrostatic'); ax[0].legend()
plt.tight_layout(); plt.show()

from scipy.stats import spearmanr
rho, _ = spearmanr(df.loc[df.FACIES == 3, 'PHI_TRUE'], df.loc[df.FACIES == 3, 'SIG_EFF_MPA'])
print(f'Shale porosity vs effective stress (Spearman): {rho:.2f}  -> strongly negative = Athy compaction')

## 4. The static–dynamic mismatch is *learnable*, not constant

`E_STAT = E_DYN × f(rock quality)` where f decreases from ~0.7 (soft, shallow) to ~0.3
(competent). A fixed conversion factor would be wrong — the ML models must learn this from
paired core-log data (the `IS_CORE` rows).

In [ ]:
r = df.E_STAT / df.E_DYN
print(f'E_stat/E_dyn: mean={r.mean():.2f}  sd={r.std():.2f}  range=[{r.min():.2f}, {r.max():.2f}]')
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df.E_DYN, r, s=2, c=df.PHI_TRUE, cmap='plasma')
ax.set_xlabel('E_dynamic (GPa)'); ax.set_ylabel('E_static / E_dynamic'); plt.show()

## 5. Core plugs — the paired training data

In [ ]:
core = df[df.IS_CORE == 1]
print(f'{len(core)} core plugs ({100*len(core)/len(df):.1f}% of depths), per well:')
print(core.groupby('WELL').size())
print('\nCore plugs by facies:'); print(core.FACIES.value_counts().rename({0:'sand',1:'shaly sand',2:'sandy shale',3:'shale'}))